# Notebook 1 — Neo4j Knowledge Graph: Data Ingestion
**Layer:** Foundation · **Scope:** hdb_feature_table_20260412.csv → Neo4j nodes & relationships  
**Inputs:** `02_feature_layer/training/outputs/hdb_feature_table_20260412.csv`, `feature_metadata_20260403.json`  
**Outputs:** 260,699 `:Flat` nodes · 9,710 `:Block` nodes · 26 `:Town` nodes · `NEAR_*` relationships

## 1.1 Install & import dependencies

In [10]:
# Run once — install required packages
# !pip install neo4j pandas tqdm
# It is is first time running, uncomment below line to install neo4j driver in notebook
# %pip install neo4j  
import pandas as pd
import json
from pathlib import Path
from neo4j import GraphDatabase
from tqdm import tqdm
ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
#FEATURE_DIR = Path("02_feature_layer/training/outputs")
CSV_PATH    = FEATURE_DIR / "hdb_feature_table_20260412.csv"
META_PATH   = FEATURE_DIR / "feature_metadata_20260412.json"

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"   # <-- update before running

print("CSV_PATH: ",CSV_PATH)
print("META_PATH: ",META_PATH)
print("Paths exist:", CSV_PATH.exists(), META_PATH.exists())


CSV_PATH:  D:\Master Degree\Projects\Transparent_AI\PropertyLens\02_feature_layer\training\outputs\hdb_feature_table_20260412.csv
META_PATH:  D:\Master Degree\Projects\Transparent_AI\PropertyLens\02_feature_layer\training\outputs\feature_metadata_20260412.json
Paths exist: True True


## 1.2 Load feature metadata & preview CSV

In [11]:
with open(META_PATH) as f:
    meta = json.load(f)

# Load a small sample first to inspect schema
sample = pd.read_csv(CSV_PATH, nrows=5)
print("Columns:", sample.columns.tolist())
print("Shape sample:", sample.shape)
sample.head()

Columns: ['resale_price', 'transaction_year', 'level_mid', 'lease_remaining_years', 'floor_area_sqm', 'room_count', 'dist_to_mrt_m', 'orientation_score', 'dist_to_highway_m', 'dist_to_foodcourt_m', 'dist_to_nearest_mall_m', 'mall_count_3km', 'mall_weighted_access_3km', 'dist_to_nearest_school_m', 'school_count_1km', 'primary_school_quality_1km_weighted', 'primary_school_top_quality_1km', 'primary_school_count_1km', 'address_key', 'town_ANG MO KIO', 'town_BEDOK', 'town_BISHAN', 'town_BUKIT BATOK', 'town_BUKIT MERAH', 'town_BUKIT PANJANG', 'town_BUKIT TIMAH', 'town_CENTRAL AREA', 'town_CHOA CHU KANG', 'town_CLEMENTI', 'town_GEYLANG', 'town_HOUGANG', 'town_JURONG EAST', 'town_JURONG WEST', 'town_KALLANG/WHAMPOA', 'town_MARINE PARADE', 'town_PASIR RIS', 'town_PUNGGOL', 'town_QUEENSTOWN', 'town_SEMBAWANG', 'town_SENGKANG', 'town_SERANGOON', 'town_TAMPINES', 'town_TOA PAYOH', 'town_WOODLANDS', 'town_YISHUN', 'flat_type_1 ROOM', 'flat_type_2 ROOM', 'flat_type_3 ROOM', 'flat_type_4 ROOM', 'fla

,resale_price,transaction_year,level_mid,lease_remaining_years,floor_area_sqm,room_count,dist_to_mrt_m,orientation_score,dist_to_highway_m,dist_to_foodcourt_m,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,255000.0,2015,8.0,70,60.0,3.0,1176.102710,1.0,605.995848,1040.523111,...,False,False,False,False,False,False,False,False,False,False
1,275000.0,2015,2.0,65,68.0,3.0,2557.686258,1.0,920.428645,1031.332729,...,False,True,False,False,False,False,False,False,False,False
2,285000.0,2015,2.0,64,69.0,3.0,1356.934758,1.0,745.459640,938.963637,...,False,True,False,False,False,False,False,False,False,False
3,290000.0,2015,2.0,63,68.0,3.0,2904.330513,1.0,1485.511064,567.906697,...,False,True,False,False,False,False,False,False,False,False
4,290000.0,2015,8.0,64,68.0,3.0,2891.203421,1.0,1267.533033,1034.040306,...,False,True,False,False,False,False,False,False,False,False


## 1.3 Define decoder helpers — one-hot → string

In [12]:
TOWN_COLS      = [c for c in sample.columns if c.startswith("town_")]
FLAT_TYPE_COLS = [c for c in sample.columns if c.startswith("flat_type_")]
FLAT_MDL_COLS  = [c for c in sample.columns if c.startswith("flat_model_")]

CORE_COLS = [
    "resale_price", "transaction_year", "level_mid", "lease_remaining_years",
    "floor_area_sqm", "room_count", "dist_to_mrt_m", "orientation_score",
    "dist_to_highway_m", "dist_to_foodcourt_m", "dist_to_nearest_mall_m",
    "mall_count_3km", "mall_weighted_access_3km", "dist_to_nearest_school_m",
    "school_count_1km", "primary_school_quality_1km_weighted",
    "primary_school_top_quality_1km", "primary_school_count_1km", "address_key"
]

def decode_onehot(row, cols, prefix):
    """Return the string value for a one-hot encoded group."""
    for c in cols:
        if row[c] == 1:
            return c.replace(prefix, "")
    return "UNKNOWN"

print(f"Town columns: {len(TOWN_COLS)}")
print(f"Flat type columns: {len(FLAT_TYPE_COLS)}")
print(f"Flat model columns: {len(FLAT_MDL_COLS)}")

Town columns: 26
Flat type columns: 7
Flat model columns: 21


## 1.4 Connect to Neo4j and create constraints

In [13]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

CONSTRAINTS = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (f:Flat)  REQUIRE f.address_key IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (b:Block) REQUIRE b.block_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Town)  REQUIRE t.name IS UNIQUE",
]

with driver.session() as session:
    for cql in CONSTRAINTS:
        session.run(cql)
        print("Applied:", cql[:60])

print("\nConstraints created successfully.")

Applied: CREATE CONSTRAINT IF NOT EXISTS FOR (f:Flat)  REQUIRE f.addr
Applied: CREATE CONSTRAINT IF NOT EXISTS FOR (b:Block) REQUIRE b.bloc
Applied: CREATE CONSTRAINT IF NOT EXISTS FOR (t:Town)  REQUIRE t.name

Constraints created successfully.


## 1.5 Batch ingest — MERGE :Town, :Block, CREATE :Flat

In [14]:
CHUNK_SIZE = 5000

UPSERT_TOWN = """
MERGE (t:Town {name: $town})
"""

UPSERT_BLOCK = """
MERGE (b:Block {block_id: $block_id})
ON CREATE SET b.address_key = $address_key, b.town = $town
"""

CREATE_FLAT = """
MERGE (f:Flat {address_key: $address_key})
SET f += $props
MERGE (b:Block {block_id: $block_id})
MERGE (t:Town  {name: $town})
MERGE (f)-[:IN_BLOCK]->(b)
MERGE (b)-[:IN_TOWN]->(t)
"""

total_rows  = 0
flat_errors = 0

reader = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)

with driver.session() as session:
    for chunk in tqdm(reader, desc="Ingesting chunks"):
        records = []
        for _, row in chunk.iterrows():
            town       = decode_onehot(row, TOWN_COLS, "town_")
            flat_type  = decode_onehot(row, FLAT_TYPE_COLS, "flat_type_")
            flat_model = decode_onehot(row, FLAT_MDL_COLS, "flat_model_")
            block_id   = str(row["address_key"]).split("_")[0]

            props = {col: float(row[col]) if isinstance(row[col], float) else row[col]
                     for col in CORE_COLS if col != "address_key"}
            props["flat_type"]  = flat_type
            props["flat_model"] = flat_model
            props["town"]       = town

            records.append({
                "address_key": str(row["address_key"]),
                "block_id":    block_id,
                "town":        town,
                "props":       props
            })

        # Batch write with UNWIND
        session.run(
            "UNWIND $records AS r "
            "MERGE (f:Flat {address_key: r.address_key}) SET f += r.props "
            "MERGE (b:Block {block_id: r.block_id}) "
            "MERGE (t:Town {name: r.town}) "
            "MERGE (f)-[:IN_BLOCK]->(b) "
            "MERGE (b)-[:IN_TOWN]->(t)",
            records=records
        )
        total_rows += len(chunk)

print(f"\nIngested {total_rows:,} rows. Errors: {flat_errors}")

Ingesting chunks: 53it [01:38,  1.87s/it]


Ingested 260,699 rows. Errors: 0


## 1.6 Create NEAR_* relationships from distance columns

In [15]:
REL_QUERY = """
UNWIND $records AS r
MATCH (f:Flat {address_key: r.address_key})
MERGE (f)-[:NEAR_MRT    {distance_m: r.dist_to_mrt_m}]->(f)
MERGE (f)-[:NEAR_HAWKER {distance_m: r.dist_to_foodcourt_m}]->(f)
MERGE (f)-[:NEAR_MALL   {distance_m: r.dist_to_nearest_mall_m,
                          count_3km: r.mall_count_3km,
                          weighted_access: r.mall_weighted_access_3km}]->(f)
MERGE (f)-[:NEAR_SCHOOL {distance_m: r.dist_to_nearest_school_m,
                          count_1km: r.school_count_1km,
                          quality_weighted: r.primary_school_quality_1km_weighted,
                          top_quality: r.primary_school_top_quality_1km,
                          count_primary_1km: r.primary_school_count_1km}]->(f)
"""
# NOTE: POI nodes are virtual — no named :MRTStation/:Mall/:School nodes
# exist in source data. Distance values are properties on self-referential
# relationships hanging off :Flat nodes only.

reader2 = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE,
    usecols=["address_key","dist_to_mrt_m","dist_to_foodcourt_m",
             "dist_to_nearest_mall_m","mall_count_3km","mall_weighted_access_3km",
             "dist_to_nearest_school_m","school_count_1km",
             "primary_school_quality_1km_weighted",
             "primary_school_top_quality_1km","primary_school_count_1km"])

with driver.session() as session:
    for chunk in tqdm(reader2, desc="Creating NEAR_* rels"):
        records = chunk.fillna(0).to_dict("records")
        session.run(REL_QUERY, records=records)

print("NEAR_* relationships created.")

Creating NEAR_* rels: 53it [00:43,  1.21it/s]

NEAR_* relationships created.


## 1.7 Validation — node & relationship counts

In [16]:
CHECKS = {
    "Flat nodes":        "MATCH (f:Flat)  RETURN count(f) AS n",
    "Block nodes":       "MATCH (b:Block) RETURN count(b) AS n",
    "Town nodes":        "MATCH (t:Town)  RETURN count(t) AS n",
    "IN_BLOCK rels":     "MATCH ()-[:IN_BLOCK]->()  RETURN count(*) AS n",
    "IN_TOWN rels":      "MATCH ()-[:IN_TOWN]->()   RETURN count(*) AS n",
    "NEAR_MRT rels":     "MATCH ()-[:NEAR_MRT]->()  RETURN count(*) AS n",
    "NEAR_SCHOOL rels":  "MATCH ()-[:NEAR_SCHOOL]->() RETURN count(*) AS n",
}

with driver.session() as session:
    for label, q in CHECKS.items():
        n = session.run(q).single()["n"]
        status = "OK" if n > 0 else "WARN"
        print(f"[{status}] {label}: {n:,}")

driver.close()
print("\nNotebook 1 complete.")

[OK] Flat nodes: 9,710
[OK] Block nodes: 9,710
[OK] Town nodes: 26
[OK] IN_BLOCK rels: 9,710
[OK] IN_TOWN rels: 9,710
[OK] NEAR_MRT rels: 9,710
[OK] NEAR_SCHOOL rels: 9,710

Notebook 1 complete.
